# Retina Age — data preparation (run once)

Downloads the dataset from HuggingFace and builds the 224x224 cache, then leaves
it in `/kaggle/working/retina224/` so it can be saved as a reusable dataset.

**Why bother:** the originals are ~10GB and ~1900x1900 px. Decoding files that big
was the bottleneck that made an epoch take 10 minutes instead of 17 seconds. Resizing
once here means every later notebook attaches ~200MB of ready-to-use images and never
downloads or resizes anything again.

**Settings for this notebook (right-hand panel):**
- Internet: **On** (needed to reach HuggingFace)
- Accelerator: **None** — this step is CPU only, don't spend GPU quota on it

**When it finishes:** Save Version, then in your training notebook add this notebook's
output as a data source. After that you never run this notebook again.


In [ ]:
!pip install -q huggingface_hub


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # avoids the download hang at ~96%

from pathlib import Path
OUT = Path('/kaggle/working/retina224')
OUT.mkdir(parents=True, exist_ok=True)
print('output ->', OUT)


## 1. Download the originals


In [ ]:
from huggingface_hub import snapshot_download

raw = Path(snapshot_download(
    repo_id='ramankamran/retina-age-analysis',
    repo_type='dataset',
    local_dir='/kaggle/temp/retina_raw',
    allow_patterns=['images/*', 'splits/*'],
))
print(raw)
print('images:', len(list((raw/'images').glob('*.jpg'))))


## 2. Copy the split CSVs and build the 224 cache

`draft()` lets libjpeg decode straight to a small size (~2.6x faster than decoding
full size then shrinking), and a thread pool uses both cores, so this takes a few
minutes rather than fifteen.


In [ ]:
import shutil
shutil.copytree(raw/'splits', OUT/'splits', dirs_exist_ok=True)
print('splits copied:', [p.name for p in (OUT/'splits').iterdir()])


In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from tqdm.auto import tqdm

SIZE = 224
dst_dir = OUT / f'images_{SIZE}'
dst_dir.mkdir(exist_ok=True)

def resize_one(src):
    dst = dst_dir / src.name
    if dst.exists():
        return True
    try:
        im = Image.open(src)
        im.draft('RGB', (SIZE, SIZE))
        im = im.convert('RGB').resize((SIZE, SIZE))
        tmp = dst.with_suffix('.tmp')
        im.save(tmp, 'JPEG', quality=90)   # format explicit: '.tmp' means nothing to PIL
        tmp.replace(dst)                   # atomic, so an interrupt can't leave a half file
        return True
    except Exception as e:
        print('skipped', src.name, e)
        return False

todo = sorted((raw/'images').glob('*.jpg'))
with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as pool:
    ok = list(tqdm(pool.map(resize_one, todo), total=len(todo), desc='resizing'))

assert all(ok), f'{ok.count(False)} images failed - do not save this version'
(dst_dir / '.complete').touch()   # marker the loader checks
print(f'{len(list(dst_dir.glob("*.jpg")))} images cached')


## 3. Check the size and sanity-check one image


In [ ]:
size_mb = sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 1e6
print(f'total output: {size_mb:.0f} MB   (originals were ~10,000 MB)')

import pandas as pd
df = pd.read_csv(OUT/'splits'/'train.csv')
one = dst_dir / f"{df.iloc[0]['image_id']}.jpg"
im = Image.open(one)
print('sample:', one.name, im.size, im.mode)
im


## Done

Now: **Save Version** (Save & Run All). When it finishes, the folder `retina224/`
becomes this notebook's output, and the training notebook can attach it as a data
source. Nothing here needs to run again.
